# 04. DFFreq 학습 실행 (Colab A100)

DFFreq 공식 저장소를 클론해 직접 구축한 8만 장 데이터셋으로 학습한 노트북.

**해결한 문제 — Google Drive I/O 병목**

8만 장을 Drive에 마운트한 채로 학습하면 데이터 로딩에서 병목이 걸려 GPU가 놀았다.
전체 데이터셋을 Colab 로컬 디스크(`/content/local_dataset`)로 먼저 복사한 뒤 학습해
에포크당 소요 시간을 크게 줄였다. 이후 재사용을 위해 `tar.gz`로 묶어 Drive에 보관했다.

- 모델: [HorizonTEL/DFFreq-main](https://github.com/HorizonTEL/DFFreq-main) (Yan et al., TIFS 2026)
- 환경: Colab A100 · batch 128 · `num_threads` 조정으로 DataLoader 교착 해결


In [ ]:
import os

# 1. 코랩 기본 작업 경로로 이동
%cd /content

# 2. 논문 공식 깃허브 저장소 복사 (Clone)
!git clone https://github.com/HorizonTEL/DFFreq-main.git

# 3. 코드 폴더로 이동
%cd /content/DFFreq-main

# 4. 다운로드된 파일 및 폴더 구조 확인
!ls -l

/content
Cloning into 'DFFreq-main'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 44 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Resolving deltas: 100% (9/9), done.
/content/DFFreq-main
total 60
drwxr-xr-x 2 root root 4096 Apr 26 15:21 checkpoints
drwxr-xr-x 2 root root 4096 Apr 26 15:21 data
-rw-r--r-- 1 root root 8360 Apr 26 15:21 log_test.log
drwxr-xr-x 2 root root 4096 Apr 26 15:21 networks
drwxr-xr-x 2 root root 4096 Apr 26 15:21 options
... (출력 6줄 생략)


In [ ]:
%cd /content/DFFreq-main

# 1. 논문 필수 라이브러리 설치 (A100 환경에 맞게 세팅)
print("📦 필수 라이브러리 설치 중...")
!pip install -r requirements.txt -q

# 2. 대망의 학습 옵션 리스트 스캔!
print("\n🔍 학습 명령어 옵션 스캔 결과:")
!python train.py --help

/content/DFFreq-main
📦 필수 라이브러리 설치 중...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.5.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.5 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.5 which is incompatible.

🔍 학습 명령어 옵션 스캔 결과:
usage: train.py [-h] [--mode MODE] [--arch ARCH] [--rz_interp RZ_INTERP]
                [--blur_prob BLUR_PROB] [--blur_sig BLUR_SIG]
                [--jpg_prob JPG_PROB] [--jpg_method JPG_METHOD]
                [--jpg_qual JPG_QUAL] [--dataroot DATAROOT]
                [--classes CLASSES] [--class_bal] [--batch_size BATCH_SIZE]
                [--loadSize LOADSIZE] [--cropSize CROPSIZE]
... (출력 93줄 생략)


In [ ]:
import os
import shutil

# 1. 모델이 요구하는 폴더 구조(dataroot/train/클래스)로 변경
base_dir = '/content/drive/MyDrive/WayV_Detection_Train'
train_dir = os.path.join(base_dir, 'train')

if not os.path.exists(train_dir):
    os.makedirs(train_dir)
    # Real과 Fake 폴더를 train 폴더 안으로 이동
    for folder in ['Real', 'Fake']:
        src = os.path.join(base_dir, folder)
        dst = os.path.join(train_dir, folder)
        if os.path.exists(src):
            shutil.move(src, dst)
    print("디렉토리 구조 변경 완료: /train 하위로 편입되었습니다.")

# 2. 작업 디렉토리 이동 및 A100 최적화 학습 시작
%cd /content/DFFreq-main

!python train.py \
  --name WayV_A100_v1 \
  --dataroot /content/drive/MyDrive/WayV_Detection_Train \
  --train_split train \
  --batch_size 128 \
  --num_threads 8 \
  --checkpoints_dir /content/drive/MyDrive/WayV_Detection_Train/checkpoints

디렉토리 구조 변경 완료: /train 하위로 편입되었습니다.
/content/DFFreq-main
----------------- Options ---------------
                     arch: resnet50                      
               batch_size: 128                           
                    beta1: 0.9                           
                blur_prob: 0.5                           
                 blur_sig: 0.0,3.0                       
          checkpoints_dir: /content/drive/MyDrive/WayV_Detection_Train/checkpoints	[default: ./checkpoints]
                class_bal: False                         
                  classes:                               
           continue_train: False                         
                 cropSize: 256                           
                 data_aug: False                         
... (출력 62줄 생략)


In [ ]:
import os
import shutil

base_dir = '/content/drive/MyDrive/WayV_Detection_Train'
train_dir = os.path.join(base_dir, 'train')
os.makedirs(train_dir, exist_ok=True)

print("🕵️‍♂️ WayV_Detection_Train 폴더 전체 수색을 시작합니다...\n")

# 1. 폴더 구석구석 뒤져서 Real, Fake 찾기
found_real = None
found_fake = None

for root, dirs, files in os.walk(base_dir):
    for d in dirs:
        if d.lower() in ['real', '0_real']:
            found_real = os.path.join(root, d)
        if d.lower() in ['fake', '1_fake']:
            found_fake = os.path.join(root, d)

print(f"📍 수색 결과 - Real 경로: {found_real}")
print(f"📍 수색 결과 - Fake 경로: {found_fake}\n")

# 2. 찾은 폴더를 올바른 위치(train/0_real, train/1_fake)로 강제 이동 및 이름 변경
if found_real:
    target_real = os.path.join(train_dir, '0_real')
    if found_real != target_real:
        os.rename(found_real, target_real)
        print("🚚 [성공] Real 폴더를 '0_real'로 고정하여 train 안으로 옮겼습니다!")
else:
    print("🚨 [경고] Real 폴더를 찾지 못했습니다! 드라이브 휴지통이나 경로 이름을 다시 확인해야 합니다.")

if found_fake:
    target_fake = os.path.join(train_dir, '1_fake')
    if found_fake != target_fake:
        os.rename(found_fake, target_fake)
        print("🚚 [성공] Fake 폴더를 '1_fake'로 고정하여 train 안으로 옮겼습니다!")
else:
    print("🚨 [경고] Fake 폴더를 찾지 못했습니다!")

# 3. 최종 상태 점검
print("\n✅ 현재 train 폴더 내부 상태: ", os.listdir(train_dir))

🕵️‍♂️ WayV_Detection_Train 폴더 전체 수색을 시작합니다...

📍 수색 결과 - Real 경로: None
📍 수색 결과 - Fake 경로: None

🚨 [경고] Real 폴더를 찾지 못했습니다! 드라이브 휴지통이나 경로 이름을 다시 확인해야 합니다.
🚨 [경고] Fake 폴더를 찾지 못했습니다!

✅ 현재 train 폴더 내부 상태:  []


In [ ]:
# 1. 코랩 드라이브 캐시 강제 새로고침 (웹사이트와 동기화)
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

# 2. A100 풀악셀 훈련 시작!
%cd /content/DFFreq-main

!python train.py \
  --name WayV_A100_v1 \
  --dataroot /content/drive/MyDrive/WayV_Detection_Train \
  --train_split train \
  --batch_size 128 \
  --num_threads 8 \
  --checkpoints_dir /content/drive/MyDrive/WayV_Detection_Train/checkpoints

Drive not mounted, so nothing to flush and unmount.


ValueError: Mountpoint must not already contain files


In [ ]:
import os
from google.colab import drive

# 1. 가짜 드라이브가 길을 막고 있다면 옆으로 치워두기 (백업)
if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    print("🚨 길을 막고 있는 '가짜 드라이브' 폴더 발견! [fake_drive_backup]으로 이름을 변경합니다.")
    os.rename('/content/drive', '/content/fake_drive_backup')

# 2. 깨끗해진 자리에 '진짜' 구글 드라이브 마운트
print("🔄 진짜 구글 드라이브를 연결합니다...")
drive.mount('/content/drive')

# 3. 대망의 A100 풀악셀 훈련 시작!
%cd /content/DFFreq-main

!python train.py \
  --name WayV_A100_v1 \
  --dataroot /content/drive/MyDrive/WayV_Detection_Train \
  --train_split train \
  --batch_size 128 \
  --num_threads 8 \
  --checkpoints_dir /content/drive/MyDrive/WayV_Detection_Train/checkpoints

🚨 길을 막고 있는 '가짜 드라이브' 폴더 발견! [fake_drive_backup]으로 이름을 변경합니다.
🔄 진짜 구글 드라이브를 연결합니다...
Mounted at /content/drive
/content/DFFreq-main
----------------- Options ---------------
                     arch: resnet50                      
               batch_size: 128                           
                    beta1: 0.9                           
                blur_prob: 0.5                           
                 blur_sig: 0.0,3.0                       
          checkpoints_dir: /content/drive/MyDrive/WayV_Detection_Train/checkpoints	[default: ./checkpoints]
                class_bal: False                         
                  classes:                               
           continue_train: False                         
... (출력 64줄 생략)


In [ ]:
import os

real_dir = '/content/drive/MyDrive/WayV_Detection_Train/train/0_real'
fake_dir = '/content/drive/MyDrive/WayV_Detection_Train/train/1_fake'

print("🔍 코랩의 드라이브 시력을 검사합니다...\n")

def check_folder(folder_path, name):
    if not os.path.exists(folder_path):
        print(f"❌ {name} 폴더 자체가 아직 코랩에 안 보입니다. (동기화 중...)")
        return

    items = os.listdir(folder_path)
    print(f"📂 {name} 폴더 인식 완료!")

    if len(items) == 0:
        print(f"  ➡️ 텅 비어있습니다! (사진들이 아직 구글 서버에서 코랩으로 넘어오는 중입니다. 1~2분 대기!)")
    elif len(items) < 10:
        print(f"  ➡️ ⚠️ 내용물이 너무 적습니다! 혹시 폴더 안에 '폴더'를 겹쳐 넣으셨나요?\n      안에 들어있는 것: {items}")
    else:
        # 사진 파일 개수 대략적으로 세기
        png_count = sum(1 for f in items if f.endswith('.png'))
        jpg_count = sum(1 for f in items if f.endswith('.jpg'))
        print(f"  ➡️ ✅ 사진들이 쏟아져 들어왔습니다! (대략 PNG: {png_count}장, JPG: {jpg_count}장 등)")
        if len(items) > 1000:
            print("  🎉 완벽하게 동기화되었습니다! 이제 훈련을 시작해도 좋습니다.")

check_folder(real_dir, '0_real')
print("-" * 50)
check_folder(fake_dir, '1_fake')

🔍 코랩의 드라이브 시력을 검사합니다...

📂 0_real 폴더 인식 완료!
  ➡️ ✅ 사진들이 쏟아져 들어왔습니다! (대략 PNG: 40000장, JPG: 0장 등)
  🎉 완벽하게 동기화되었습니다! 이제 훈련을 시작해도 좋습니다.
--------------------------------------------------
📂 1_fake 폴더 인식 완료!
  ➡️ ⚠️ 내용물이 너무 적습니다! 혹시 폴더 안에 '폴더'를 겹쳐 넣으셨나요?
      안에 들어있는 것: ['StyleGAN2', 'StyleGAN3']


In [ ]:
# A100 풀악셀 훈련 시작!
%cd /content/DFFreq-main

!python train.py \
  --name WayV_A100_v1 \
  --dataroot /content/drive/MyDrive/WayV_Detection_Train \
  --train_split train \
  --batch_size 128 \
  --num_threads 8 \
  --checkpoints_dir /content/drive/MyDrive/WayV_Detection_Train/checkpoints

/content/DFFreq-main
----------------- Options ---------------
                     arch: resnet50                      
               batch_size: 128                           
                    beta1: 0.9                           
                blur_prob: 0.5                           
                 blur_sig: 0.0,3.0                       
          checkpoints_dir: /content/drive/MyDrive/WayV_Detection_Train/checkpoints	[default: ./checkpoints]
                class_bal: False                         
                  classes:                               
           continue_train: False                         
                 cropSize: 256                           
                 data_aug: False                         
                 dataroot: /content/drive/MyDrive/WayV_Detection_Train	[default: ./dataset/]
... (출력 62줄 생략)


In [ ]:
# A100 풀악셀 훈련 시작! (Validation 폴더 우회 패치 적용)
%cd /content/DFFreq-main

!python train.py \
  --name WayV_A100_v1 \
  --dataroot /content/drive/MyDrive/WayV_Detection_Train \
  --train_split train \
  --val_split train \
  --batch_size 128 \
  --num_threads 8 \
  --checkpoints_dir /content/drive/MyDrive/WayV_Detection_Train/checkpoints

/content/DFFreq-main
----------------- Options ---------------
                     arch: resnet50                      
               batch_size: 128                           
                    beta1: 0.9                           
                blur_prob: 0.5                           
                 blur_sig: 0.0,3.0                       
          checkpoints_dir: /content/drive/MyDrive/WayV_Detection_Train/checkpoints	[default: ./checkpoints]
                class_bal: False                         
                  classes:                               
           continue_train: False                         
                 cropSize: 256                           
                 data_aug: False                         
                 dataroot: /content/drive/MyDrive/WayV_Detection_Train	[default: ./dataset/]
... (출력 89줄 생략)


In [ ]:
%cd /content/DFFreq-main

!python train.py \
  --name WayV_A100_v1 \
  --dataroot /content/drive/MyDrive/WayV_Detection_Train \
  --train_split train \
  --val_split train \
  --batch_size 128 \
  --num_threads 0 \
  --checkpoints_dir /content/drive/MyDrive/WayV_Detection_Train/checkpoints

/content/DFFreq-main
----------------- Options ---------------
                     arch: resnet50                      
               batch_size: 128                           
                    beta1: 0.9                           
                blur_prob: 0.5                           
                 blur_sig: 0.0,3.0                       
          checkpoints_dir: /content/drive/MyDrive/WayV_Detection_Train/checkpoints	[default: ./checkpoints]
                class_bal: False                         
                  classes:                               
           continue_train: False                         
                 cropSize: 256                           
                 data_aug: False                         
                 dataroot: /content/drive/MyDrive/WayV_Detection_Train	[default: ./dataset/]
... (출력 67줄 생략)


In [ ]:
import os
import shutil
from tqdm.auto import tqdm

src_base = '/content/drive/MyDrive/WayV_Detection_Train/train'
dst_base = '/content/local_dataset/train'

os.makedirs(dst_base, exist_ok=True)

print("🔍 구글 드라이브의 파일 목록을 스캔하는 중... (약 2~3분 소요)")
all_files = []
for root, dirs, files in os.walk(src_base):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"📦 총 {len(all_files)}개의 파일을 코랩 초고속 SSD로 복사합니다!")

# 복사 진행 (tqdm으로 실시간 진행상황 확인)
for src_file in tqdm(all_files, desc="로컬 SSD로 데이터 전송 중"):
    rel_path = os.path.relpath(src_file, src_base)
    dst_file = os.path.join(dst_base, rel_path)
    os.makedirs(os.path.dirname(dst_file), exist_ok=True)

    if not os.path.exists(dst_file):
        shutil.copy2(src_file, dst_file)

print("\n✅ 로컬 SSD 복사 완료! 이제 빛의 속도로 훈련이 가능합니다.")

🔍 구글 드라이브의 파일 목록을 스캔하는 중... (약 2~3분 소요)
📦 총 79638개의 파일을 코랩 초고속 SSD로 복사합니다!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 8만 장 압축 시작 (화면에 아무 변화가 없어도 뒤에서 열심히 압축 중이니 기다려주세요)
!tar -czf /content/drive/MyDrive/WayV_Detection_Train/train_data.tar.gz -C /content/drive/MyDrive/WayV_Detection_Train/ train

print("✅ 압축 완료! 이제 파일 1개로 최적화되었습니다.")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
tar: train/1_fake/StyleGAN3/sg3_fake_05454.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05455.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05456.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05457.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05458.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05459.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05460.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05461.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05462.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05463.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05464.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05465.png: file changed as we read it
tar: train/1_fake/StyleGAN3/sg3_fake_05466.png: file changed as 